# AI Text Summarizer (GenAI / LLM + Gradio)
Paste an article or upload a PDF and get a **short summary**, **bullet points**, or an **"Explain like I'm 12"** version.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom. The last cell launches the Gradio app.

## Working procedure
1. **Load an open-source LLM** (Qwen2.5-3B-Instruct) on the GPU. No API key is needed.
2. **Read the input:** text from the box, or text extracted from a PDF / TXT file.
3. **Clean the text:** fix broken lines, extra spaces and hyphenated words.
4. **Count tokens:** a short document goes to the LLM directly. A long document is too big for one prompt, so it is split into chunks.
5. **Map step (long documents):** the LLM writes short notes for every chunk.
6. **Reduce step:** all the notes are combined, and the LLM writes the final output in the style you selected.
7. **Show the result** in the Gradio app together with the word count, number of LLM calls and time taken.

## Workflow diagram
```
 ┌─────────────────────┐        ┌──────────────────────┐
 │ Pasted article text │        │ Uploaded PDF / TXT   │
 └──────────┬──────────┘        └──────────┬───────────┘
            │                              ▼
            │                   ┌──────────────────────┐
            │                   │ Extract text (pypdf) │
            │                   └──────────┬───────────┘
            └──────────────┬───────────────┘
                           ▼
              ┌──────────────────────────┐
              │ Clean text + count tokens│
              └────────────┬─────────────┘
                           ▼
                 ┌───────────────────┐
                 │ Fits in one prompt?│
                 └───┬───────────┬───┘
                 YES │           │ NO
                     │           ▼
                     │  ┌────────────────────────────┐
                     │  │ Split into chunks          │
                     │  │ MAP: LLM notes per chunk   │
                     │  │ Combine all notes          │
                     │  └─────────────┬──────────────┘
                     └────────┬───────┘
                              ▼
        ┌───────────────────────────────────────────────┐
        │ REDUCE: final LLM prompt for the chosen style │
        │  • Short summary  • Bullet points  • ELI12    │
        └────────────────────┬──────────────────────────┘
                             ▼
                  ┌─────────────────────┐
                  │ Gradio app (output) │
                  └─────────────────────┘
```

## LLM techniques used
- **Prompt engineering:** a separate system prompt for each output style.
- **Few-shot prompting:** the "Explain like I'm 12" prompt includes an example of the style.
- **Grounding:** the prompts tell the model to use only facts from the text, which reduces made-up content.
- **Map-reduce summarization (chunking):** lets the model handle documents longer than its prompt.
- **Controllable generation:** length options and a temperature (creativity) slider.

## Step 1: Install libraries and check the GPU

In [ ]:
!pip install -q gradio pypdf accelerate

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## Step 2: Load the LLM
This downloads about 6 GB the first time (2-3 minutes). If you get a GPU out-of-memory error, change the name to `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
llm.eval()
print("Model loaded on:", llm.device)

## Step 3: A helper function to talk to the LLM

In [ ]:
def ask_llm(system_prompt, user_prompt, max_new_tokens=300, temperature=0.3):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)

    settings = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        settings.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        settings.update(do_sample=False)

    with torch.no_grad():
        output = llm.generate(**inputs, **settings)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

print(ask_llm("You are a helpful assistant.", "Say hello in one short sentence."))

## Step 4: Read and clean the input (text or PDF)

In [ ]:
import re
from pypdf import PdfReader

def read_pdf(path):
    reader = PdfReader(path)
    pages = [(page.extract_text() or "") for page in reader.pages]
    return "\n".join(pages)

def read_file(path):
    if path.lower().endswith(".pdf"):
        return read_pdf(path)
    return open(path, encoding="utf-8", errors="ignore").read()

def clean_text(text):
    text = re.sub(r"-\n(\w)", r"\1", text)      # join words broken across lines
    text = re.sub(r"[ \t]+", " ", text)         # extra spaces
    text = re.sub(r"\n{3,}", "\n\n", text)      # extra blank lines
    return text.strip()

def count_tokens(text):
    return len(tok(text, add_special_tokens=False)["input_ids"])

## Step 5: Split long documents into chunks
Sentences are packed together until a chunk reaches the token limit, so no sentence is cut in half.

In [ ]:
def chunk_text(text, max_tokens=1500):
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text)
    pieces = []
    for s in sentences:
        words = s.split()
        for i in range(0, len(words), 600):          # break extremely long "sentences"
            piece = " ".join(words[i:i+600])
            if piece:
                pieces.append(piece)

    lengths = [len(ids) for ids in tok(pieces, add_special_tokens=False)["input_ids"]]
    chunks, current, current_len = [], [], 0
    for piece, n in zip(pieces, lengths):
        if current and current_len + n > max_tokens:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(piece)
        current_len += n
    if current:
        chunks.append(" ".join(current))
    return chunks

## Step 6: Prompts for each output style
Each style has its own system prompt. All of them tell the model to use only the information in the text (grounding). The "Explain like I'm 12" prompt includes one example (few-shot).

In [ ]:
LENGTHS = {
    "Short":  {"sentences": "2",   "bullets": "4",  "eli": "3",   "tokens": 160},
    "Medium": {"sentences": "4",   "bullets": "6",  "eli": "5",   "tokens": 300},
    "Long":   {"sentences": "6-8", "bullets": "10", "eli": "8",   "tokens": 500},
}

GROUNDING = (
    "Use ONLY information that is in the text. Do not add facts, numbers, names or opinions "
    "that are not in the text. Write the answer directly, without an introduction like 'Here is'."
)

def style_prompt(style, length):
    L = LENGTHS[length]
    if style == "Short summary":
        return (f"You are an expert summarizer. Write a clear, accurate summary of the text in about "
                f"{L['sentences']} sentences, in one paragraph. Cover the main idea and the most important points. "
                + GROUNDING)
    if style == "Bullet points":
        return (f"You are an expert summarizer. Summarize the text as about {L['bullets']} bullet points. "
                f"Start every line with '- '. Each bullet is one short, clear sentence with one key point. "
                + GROUNDING)
    # Explain like I'm 12 (few-shot: one example of the style)
    return (f"You explain things to a 12-year-old. Explain the main ideas of the text in about {L['eli']} short "
            f"sentences, using simple everyday words and, where it helps, a friendly comparison. "
            f"Avoid difficult jargon; if a hard word is needed, explain it. " + GROUNDING + "\n\n"
            "Example of the style:\n"
            "Text: 'Photosynthesis is the process by which plants convert light energy into chemical energy stored in glucose.'\n"
            "Explanation: 'Plants are like tiny solar-powered kitchens. They catch sunlight and use it to make their own food, "
            "a sugar called glucose, which gives them the energy to grow.'")

NOTES_PROMPT = (
    "You are a careful note-taker. Read the text and write the key facts and ideas as short bullet points "
    "(maximum 8 bullets, each starting with '- '). " + GROUNDING
)

## Step 7: The summarizer (with map-reduce for long documents)
- **Short document:** one LLM call with the style prompt.
- **Long document:** *Map* = notes for each chunk, *Reduce* = combine the notes and write the final output. If the combined notes are still too long, the notes are summarized again.

In [ ]:
import time

DIRECT_LIMIT = 2500        # documents up to this many tokens go to the LLM in one prompt
MAX_DOC_TOKENS = 40000     # very long documents are cut to this size

def summarize_document(text, style="Short summary", length="Medium", temperature=0.3, progress=None):
    start = time.time()
    text = clean_text(text)
    words_in = len(text.split())
    if words_in < 30:
        return "⚠️ The text is too short to summarize. Please give at least a few sentences.", ""

    tokens_in = count_tokens(text)
    note = ""
    if tokens_in > MAX_DOC_TOKENS:
        ids = tok(text, add_special_tokens=False)["input_ids"][:MAX_DOC_TOKENS]
        text = tok.decode(ids)
        note = " (document was very long, only the first part was used)"

    calls, n_chunks = 0, 1
    final_tokens = LENGTHS[length]["tokens"]

    if count_tokens(text) <= DIRECT_LIMIT:
        if progress: progress(0.3, desc="Summarizing...")
        output = ask_llm(style_prompt(style, length), f"Text:\n{text}", final_tokens, temperature)
        calls = 1
    else:
        # MAP: notes for every chunk
        chunks = chunk_text(text)
        n_chunks = len(chunks)
        notes = []
        for i, chunk in enumerate(chunks):
            if progress: progress(0.1 + 0.7 * i / n_chunks, desc=f"Reading part {i+1} of {n_chunks}...")
            notes.append(ask_llm(NOTES_PROMPT, f"Text:\n{chunk}", 250, 0.2))
            calls += 1
        combined = "\n".join(notes)

        # If the notes are still too long, compress them again
        rounds = 0
        while count_tokens(combined) > DIRECT_LIMIT and rounds < 3:
            parts = chunk_text(combined)
            combined = "\n".join(ask_llm(NOTES_PROMPT, f"Text:\n{p}", 250, 0.2) for p in parts)
            calls += len(parts); rounds += 1

        # REDUCE: final output in the chosen style
        if progress: progress(0.9, desc="Writing the final result...")
        output = ask_llm(style_prompt(style, length),
                         f"These are notes taken from consecutive parts of one document:\n{combined}\n\n"
                         f"Write the final result for the whole document.", final_tokens, temperature)
        calls += 1

    stats = (f"Input: {words_in:,} words ({tokens_in:,} tokens){note} | Chunks: {n_chunks} | "
             f"LLM calls: {calls} | Output: {len(output.split())} words | Time: {time.time()-start:.1f}s")
    return output, stats

## Step 8: Test it on a sample article (before using the app)

In [ ]:
SAMPLE_ARTICLE = '''
Sleep and Learning: Why a Good Night Matters

Many students believe that staying up all night to study is the best way to prepare for an exam. Research on sleep suggests the opposite. During sleep, the brain moves new information from short-term memory into long-term memory, a process called memory consolidation. When a person skips sleep, this process is cut short, and the material studied the day before is much more likely to be forgotten.

Sleep happens in cycles of about ninety minutes, and each cycle contains light sleep, deep sleep and REM sleep. Deep sleep is linked to remembering facts, while REM sleep is linked to understanding concepts and solving problems. A person who sleeps only four or five hours misses a large share of both kinds of sleep.

Lack of sleep also affects attention. Tired students find it harder to concentrate, react more slowly and make more mistakes, even if they feel that they are coping well. Studies of teenagers show that those who sleep less than seven hours tend to have lower grades and more trouble managing their mood.

Experts recommend that teenagers and young adults sleep between seven and nine hours each night. Simple habits can help: going to bed at the same time every day, avoiding screens for an hour before sleeping, and keeping the bedroom dark and quiet. Caffeine late in the day can also delay sleep and should be avoided. Studying in shorter sessions spread over several days, followed by a full night of rest, is a far better plan than a single all-night effort.
'''

for style in ["Short summary", "Bullet points", "Explain like I'm 12"]:
    result, info = summarize_document(SAMPLE_ARTICLE, style, "Medium")
    print(f"===== {style} =====")
    print(result)
    print(info, "\n")

## Step 9: Launch the Gradio app
Run this cell, then open the **public link** that appears (ending in `gradio.live`), or use the app shown below the cell. Paste text or upload a PDF, choose a style and press **Summarize**.

In [ ]:
import gradio as gr

def gradio_summarize(text, file, style, length, temperature, progress=gr.Progress()):
    try:
        if file is not None:
            path = file if isinstance(file, str) else file.name
            raw = read_file(path)
            if not raw.strip():
                return "⚠️ No text could be read from this file. Scanned PDFs (images) are not supported.", ""
        else:
            raw = text or ""
        if not raw.strip():
            return "⚠️ Please paste some text or upload a PDF.", ""
        result, info = summarize_document(raw, style, length, temperature, progress)
        return result, info
    except Exception as e:
        return f"⚠️ Something went wrong: {e}", ""

with gr.Blocks(title="AI Text Summarizer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📝 AI Text Summarizer\nPaste an article or upload a PDF, then choose how you want it summarized.")

    with gr.Row():
        with gr.Column():
            text_in = gr.Textbox(label="Paste your article here", lines=14,
                                 placeholder="Paste a news article, blog post, report, notes...")
            file_in = gr.File(label="...or upload a PDF / TXT file (used instead of the text box)",
                              file_types=[".pdf", ".txt"])
            sample_btn = gr.Button("Load sample article")

        with gr.Column():
            style = gr.Radio(["Short summary", "Bullet points", "Explain like I'm 12"],
                             value="Short summary", label="Output style")
            length = gr.Radio(["Short", "Medium", "Long"], value="Medium", label="Length")
            temperature = gr.Slider(0.0, 1.0, value=0.3, step=0.1,
                                    label="Creativity (0 = most factual, 1 = more creative)")
            run_btn = gr.Button("Summarize", variant="primary")
            gr.Markdown("### Result")
            result_out = gr.Markdown()
            info_out = gr.Textbox(label="Details", interactive=False)

    sample_btn.click(lambda: SAMPLE_ARTICLE.strip(), outputs=text_in)
    run_btn.click(gradio_summarize,
                  inputs=[text_in, file_in, style, length, temperature],
                  outputs=[result_out, info_out])

demo.launch(share=True, debug=True)

## Notes
- **Scanned PDFs** (pages that are pictures) have no text to extract. Use an OCR tool first.
- **Speed:** a short article takes a few seconds. A long PDF takes longer because every chunk needs one LLM call.
- **Better quality:** use a bigger model (for example `Qwen/Qwen2.5-7B-Instruct` with 4-bit loading), or replace `ask_llm` with a call to a hosted LLM API. The rest of the code stays the same.
- **Extension ideas:** add a "Ask a question about this PDF" box (retrieval-augmented generation), multi-language output, or a download button for the summary.